# Designing a high-frequency pulse for cold-cell heating

Fast charging at subzero temperatures causes lithium plating. Slow intercalation kinetics push the anode potential below the plating threshold, and metallic lithium deposits on the particle surface instead, causing irreversible capacity loss. The mitigation is to pre-heat the cell. Pulse heating is an efficient means of doing so. A square-wave current with zero mean is applied, and the resulting ohmic heat generation warms the cell from within. Since there is no net charge throughput, the state of charge remains at its initial value.

Since the heat generation scales with the square of the current, faster heating requires larger current amplitudes. Three constraints limit how large the amplitude can be, which are,

- the terminal voltage must remain within the cell cutoffs
- the anode potential must remain above the plating threshold
- the cathode potential must remain below its upper threshold

This is where frequency of the pulse matters. At high frequency the double layer shunts the charge-transfer resistance, so the effective cell impedance is lower and a larger amplitude fits within the same voltage and electrode-potential headroom.

This notebook looks at two things:

- how cell resistance depends on pulse frequency, which is the reasoning behind running at high frequency
- for a chosen frequency and set of safety limits, a lookup table approach to find the maximum pulse amplitude that keeps the cell inside those limits

## Settings

In [46]:
from breathe_simulate import api_interface as api
from breathe_simulate.cycler import Cycler

cell_name = "SWDAGPA-D"  # platform name for the Sunwoda cell (the one with a fitted double layer)
soc = 0.4  # state of charge for the experiment [-]
T_start_degC = -20  # cell temperature at the start of the heat up [degC]
T_amb_degC = -20  # ambient temperature the cell sits in [degC]
freq_Hz = 100  # frequency of the zero-mean square wave [Hz]

t_end_s = 500  # maximum simulation time [s]
T_target_degC = 0  # simulation stops when the cell reaches this [degC]

vAnodeMin_V = 0.015  # lowest allowed anode potential [V], margin above 0 V plating
vCathodeMax_V = 4.4  # highest allowed cathode potential [V]
vMin_V = 2.8  # cell lower cutoff voltage [V]
vMax_V = 4.35  # cell upper cutoff voltage [V]
vMargin_V = 0.050  # margin kept inside both voltage cutoffs [V]
heatTransCoeff_SI = 15  # convective heat transfer to ambient [W/(m^2*K)]

lutTempBps_degC = list(
    range(-20, 41, 5)
)  # temperatures the max-current table is built at [degC]
lutSocBps = [
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
]  # states of charge the table is built at [-]
cRateBps = [
    0.25 * i for i in range(41)
]  # C-rates searched when solving for the limit [-]
lutPlotTMax_degC = 10  # highest temperature shown on the table plot [degC]

# The safety limits are collected once and passed to both the lookup table and
# the simulation, so the table that is plotted is the table the run flies.
limits = dict(
    freq_hz=freq_Hz,
    v_anode_min_v=vAnodeMin_V,
    v_cathode_max_v=vCathodeMax_V,
    v_min_v=vMin_V,
    v_max_v=vMax_V,
    v_margin_v=vMargin_V,
    temperature_breakpoints_c=lutTempBps_degC,
    soc_breakpoints=lutSocBps,
    c_rate_breakpoints=cRateBps,
)

## Utility functions

Plot helpers only, and the Breathe house colours. MATLAB keeps these at the end of a live script; Python has to define them before they are used, so they sit here.

In [47]:
import math

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Breathe plot colours. Fixed house style, not a study setting.
green = "rgb(141, 198, 63)"
grey = "rgb(115, 120, 122)"
dark = "rgb(64, 69, 71)"
yellow = "rgb(237, 199, 41)"
band = "rgb(230, 232, 235)"  # shading for out-of-limit regions


def soc_colour(t, c_low, c_mid, c_high):
    """Grey through light green to yellow across the SOC range."""
    lo, hi = (c_low, c_mid) if t <= 0.5 else (c_mid, c_high)
    f = 2 * t if t <= 0.5 else 2 * (t - 0.5)
    rgb = [a + f * (b - a) for a, b in zip(lo, hi)]
    return "rgb({:.0f}, {:.0f}, {:.0f})".format(*rgb)


def envelope_band(
    fig, t_min, lo, hi, line_colour, fill_rgba, row, col, show_legend=False
):
    """Shaded peak-to-trough band with both edges drawn on top."""
    fig.add_trace(
        go.Scatter(
            x=t_min,
            y=hi,
            mode="lines",
            name="per-cycle peak",
            line=dict(color=line_colour, width=1.8),
            showlegend=show_legend,
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=t_min,
            y=lo,
            mode="lines",
            name="per-cycle trough",
            line=dict(color=line_colour, width=1.8, dash="dash"),
            fill="tonexty",
            fillcolor=fill_rgba,
            showlegend=show_legend,
        ),
        row=row,
        col=col,
    )


def limit_band(fig, y0, y1, row, col):
    """Shading marking where the signal must not go, pushed behind the data."""
    fig.add_hrect(
        y0=y0,
        y1=y1,
        fillcolor=band,
        opacity=0.85,
        line_width=0,
        layer="below",
        row=row,
        col=col,
    )


def style_axes(fig):
    fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(0,0,0,0.12)",
        ticks="outside",
        linecolor=dark,
        color=dark,
    )
    fig.update_yaxes(
        showgrid=True,
        gridcolor="rgba(0,0,0,0.12)",
        ticks="outside",
        linecolor=dark,
        color=dark,
    )
    fig.update_layout(
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(family="Arial", size=12, color=dark),
    )

## Frequency dependence of the effective resistance

This section evaluates the frequency dependence of the effective resistance from a high-frequency pulse experiment. The effective resistance is defined as the ratio of the peak-to-peak voltage difference to the peak-to-peak pulse current. 

In [48]:
rf = api.generate_resistance_vs_frequency(
    cell_name,
    temperature_c=T_start_degC,
    soc=soc,
    c_rate=1.0,
    freq_min_hz=1.0,
    freq_max_hz=500.0,
)

print("{}, {:.1f} Ah".format(cell_name, rf.metadata["capacity_ah"]))

SWDAGPA-D, 136.4 Ah


In [49]:
r_pick_mohm = rf.at(freq_Hz)  # resistance at the frequency chosen above
r_low_mohm = rf.resistance_mohm[0]  # the low-frequency end of the sweep

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=rf.frequency_hz,
        y=rf.resistance_mohm,
        mode="lines",
        line=dict(color=green, width=2.2),
        name="total resistance",
    )
)
fig.add_trace(
    go.Scatter(
        x=[freq_Hz],
        y=[r_pick_mohm],
        mode="markers",
        marker=dict(size=10, color=dark, symbol="circle-open", line=dict(width=2)),
    )
)
fig.add_vline(
    x=freq_Hz,
    line_dash="dot",
    line_color=grey,
    annotation_text="  {:g} Hz, {:.2f} mΩ".format(freq_Hz, r_pick_mohm),
    annotation_position="top right",
)
fig.add_hline(
    y=rf.resistance_dc_mohm,
    line_dash="dot",
    line_color=grey,
    annotation_text="DC limit",
    annotation_position="top left",
)
fig.add_hline(
    y=rf.resistance_high_frequency_mohm,
    line_dash="dot",
    line_color=grey,
    annotation_text="high-frequency limit",
    annotation_position="bottom left",
)

fig.update_xaxes(
    type="log",
    title_text="Pulse frequency (Hz)",
    range=[math.log10(min(rf.frequency_hz)), math.log10(max(rf.frequency_hz))],
)
fig.update_yaxes(range=[0, 1.15 * r_low_mohm], title_text="Total cell resistance (mΩ)")
fig.update_layout(
    title="{} at {:g} °C, SOC {:.0f}%".format(cell_name, T_start_degC, 100 * soc),
    width=900,
    height=520,
    showlegend=False,
)
style_axes(fig)
fig.show()

print(
    "Resistance falls from {:.2f} mOhm at {:g} Hz to {:.2f} mOhm at {:g} Hz ({:.0f}% lower)".format(
        r_low_mohm,
        rf.frequency_hz[0],
        r_pick_mohm,
        freq_Hz,
        100 * (1 - r_pick_mohm / r_low_mohm),
    )
)

Resistance falls from 3.09 mOhm at 1 Hz to 0.59 mOhm at 100 Hz (81% lower)


The curve drops steeply and then flattens, so there is little to gain by pushing past the knee. For this cell it sits near 100 Hz, which is the frequency used from here on.

## Maximum pulse amplitude lookup table

With the frequency fixed and the voltage and electrode-potential limits specified, the next step is to determine the largest pulse amplitude that can be applied without breaching them, so that as much heating as possible is obtained from the pulse. The method `generate_pulse_heating_lut` derives this maximum C-rate analytically as a lookup table in SOC and temperature. The resulting table is plotted below.

In [50]:
lut = api.generate_pulse_heating_lut(cell_name, **limits)
lut

PulseHeatingLutResults(7 SoC x 13 T at 100 Hz)

In [51]:
max_c_rate = lut.to_dataframe("c_rate_max")  # rows: SOC, columns: temperature
shown = max_c_rate.loc[:, max_c_rate.columns <= lutPlotTMax_degC]

fig = go.Figure()
n_soc = len(max_c_rate.index)
for i, soc_bp in enumerate(max_c_rate.index):
    colour = soc_colour(
        i / max(n_soc - 1, 1), (115, 120, 122), (141, 198, 63), (237, 199, 41)
    )
    fig.add_trace(
        go.Scatter(
            x=max_c_rate.columns,
            y=max_c_rate.loc[soc_bp],
            mode="lines+markers",
            line=dict(color=colour, width=2.2),
            marker=dict(size=5, color=colour),
            name="SOC {:.1f}".format(soc_bp),
        )
    )

t_min = min(max_c_rate.columns)
fig.update_xaxes(
    range=[t_min, lutPlotTMax_degC + 0.22 * (lutPlotTMax_degC - t_min)],
    title_text="Temperature (°C)",
)
fig.update_yaxes(
    range=[0, 1.12 * shown.to_numpy().max()],
    title_text="Maximum pulse amplitude (C-rate)",
)
fig.update_layout(
    title="{}, largest safe zero-mean pulse at {:g} Hz (anode ≥ {:g} mV, cathode ≤ {:.2f} V)".format(
        cell_name, freq_Hz, 1e3 * vAnodeMin_V, vCathodeMax_V
    ),
    width=900,
    height=560,
    legend_title="",
)
style_axes(fig)
fig.show()

The maximum pulse amplitude increases with temperature as the cell resistance falls. It decreases with state of charge, since the anode open-circuit potential approaches the plating threshold at high SOC, leaving less headroom for the anode overpotential.

## Example of high-frequency pulse heating

In this section, high-frequency pulse heating is simulated at the prescribed SOC and starting temperature. The model runs in closed loop: the SOC and temperature computed by the model are fed back to the lookup table, which sets the amplitude of the applied square wave, so the permitted current is updated continuously as the cell warms. The run terminates when the target temperature is reached, or when the simulation time exceeds `t_end_s`.

In [52]:
import time

cycler = Cycler(selected_unit="C", cell_capacity=lut.metadata["capacity_ah"])
protocol = cycler.pulse_heating(
    **limits, duration_s=t_end_s, target_temperature_c=T_target_degC
)

print("Simulating at {:g} Hz. This takes a while.".format(freq_Hz))
t_run = time.perf_counter()

api._use_async_endpoints = True
try:
    results = api.run_sim(
        base_battery=cell_name,
        cycler=protocol,
        initialSoC=soc,
        initialTemperature_degC=T_start_degC,
        ambientTemperature_degC=T_amb_degC,
        heatTransferCoefficient=heatTransCoeff_SI,
    )
finally:
    api._use_async_endpoints = False

print("Done in {:.1f} min of wall time.".format((time.perf_counter() - t_run) / 60))

Simulating at 100 Hz. This takes a while.
Done in 9.2 min of wall time.


A run at high frequency spans tens of thousands of cycles, too many to resolve individually, so current, voltage, anode potential and cathode potential are plotted as the per-cycle maxima and minima. Shaded regions denote the excluded operating envelope.

In [53]:
trace = results.dynamic_data["Baseline"]
t_min = [t / 60 for t in trace["Time [s]"]]

fig = make_subplots(
    rows=2,
    cols=6,
    specs=[
        [{"colspan": 3}, None, None, {"colspan": 3}, None, None],
        [{"colspan": 2}, None, {"colspan": 2}, None, {"colspan": 2}, None],
    ],
    subplot_titles=(
        "Pulse current",
        "Cell temperature",
        "Cell voltage (cutoffs {:.2f} / {:.2f} V)".format(vMin_V, vMax_V),
        "Anode potential (threshold {:g} mV)".format(1e3 * vAnodeMin_V),
        "Cathode potential (threshold {:.2f} V)".format(vCathodeMax_V),
    ),
    horizontal_spacing=0.07,
    vertical_spacing=0.14,
)

envelope_band(
    fig,
    t_min,
    trace["Current [A] (min)"],
    trace["Current [A] (max)"],
    dark,
    "rgba(115, 120, 122, 0.22)",
    row=1,
    col=1,
    show_legend=True,
)

fig.add_trace(
    go.Scatter(
        x=t_min,
        y=trace["Cell temperature [°C]"],
        mode="lines",
        line=dict(color=green, width=2.4),
        showlegend=False,
    ),
    row=1,
    col=4,
)
fig.add_hline(
    y=T_target_degC,
    line_dash="dot",
    line_color=dark,
    annotation_text="target",
    row=1,
    col=4,
)

envelope_band(
    fig,
    t_min,
    trace["Voltage [V] (min)"],
    trace["Voltage [V] (max)"],
    green,
    "rgba(141, 198, 63, 0.22)",
    row=2,
    col=1,
)
limit_band(fig, vMax_V, vMax_V + 1, row=2, col=1)
limit_band(fig, vMin_V - 1, vMin_V, row=2, col=1)
fig.update_yaxes(range=[vMin_V - 0.08, vMax_V + 0.08], row=2, col=1)

anode_lo = trace["Negative electrode potential [V] (min)"]
anode_hi = trace["Negative electrode potential [V] (max)"]
envelope_band(
    fig, t_min, anode_lo, anode_hi, green, "rgba(141, 198, 63, 0.22)", row=2, col=3
)
limit_band(fig, vAnodeMin_V - 1, vAnodeMin_V, row=2, col=3)
fig.update_yaxes(
    range=[min(0, min(anode_lo)) - 0.01, max(anode_hi) + 0.02], row=2, col=3
)

cathode_lo = trace["Positive electrode potential [V] (min)"]
cathode_hi = trace["Positive electrode potential [V] (max)"]
envelope_band(
    fig, t_min, cathode_lo, cathode_hi, green, "rgba(141, 198, 63, 0.22)", row=2, col=5
)
limit_band(fig, vCathodeMax_V, vCathodeMax_V + 1, row=2, col=5)
fig.update_yaxes(
    range=[min(cathode_lo) - 0.02, max(vCathodeMax_V, max(cathode_hi)) + 0.02],
    row=2,
    col=5,
)

for col in (1, 4):
    fig.update_xaxes(title_text="Time (min)", row=1, col=col)
for col in (1, 3, 5):
    fig.update_xaxes(title_text="Time (min)", row=2, col=col)
fig.update_yaxes(title_text="Current (A)", row=1, col=1)
fig.update_yaxes(title_text="Cell temperature (°C)", row=1, col=4)
fig.update_yaxes(title_text="Cell voltage (V)", row=2, col=1)
fig.update_yaxes(title_text="Anode potential (V)", row=2, col=3)
fig.update_yaxes(title_text="Cathode potential (V)", row=2, col=5)

fig.update_layout(
    title="{}, SOC {:.0f}%, {:g} Hz, starting at {:g} °C".format(
        cell_name, 100 * soc, freq_Hz, T_start_degC
    ),
    width=1300,
    height=740,
)
style_axes(fig)
fig.show()

temperature = trace["Cell temperature [°C]"]
duration_s = trace["Time [s]"][-1]
reached = temperature[-1] >= T_target_degC - 1e-3
rate_per_min = (temperature[-1] - temperature[0]) / (duration_s / 60)

print(
    "\nTarget temperature {:+.1f} degC {} {:.1f} min from {:+.1f} degC, averaging {:.2f} degC/min".format(
        T_target_degC,
        "reached in" if reached else "not reached within",
        duration_s / 60,
        temperature[0],
        rate_per_min,
    )
)


Target temperature +0.0 degC not reached within 5.4 min from -20.0 degC, averaging 3.67 degC/min


As the cell warms, the lookup table permits a progressively larger current amplitude, so the current envelope widens and the heating rate increases. None of the cell voltage, anode potential or cathode potential enters its shaded region at any point in the run, so all three limits are respected throughout.